# 06 — Evaluation and guardrails

**Goal.** The chapter most agentic-AI tutorials skip. Three things that turn a demo into something you'd actually deploy:

1. **Trace every model call.** Replay them later without re-running the LLM.
2. **Measure failure modes.** Count how often each one happened across your demo runs.
3. **Add guardrails.** Budget caps, refusal patterns, and a human-in-the-loop confirmation gate for any tool that mutates state.

Nothing in this notebook needs new infrastructure. Tracing is a JSONL file. Replay is `json.load`. Guardrails are if-statements. The point is to show that responsible agentic AI is *not* gated on heavy MLOps tooling — it's gated on caring enough to write the if-statements.

## 1. Setup

In [ ]:
import json, nbformat, ollama
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import duckdb

for nb_path in ["01_tools_not_agents.ipynb", "04_memory.ipynb"]:
    nb = nbformat.read(Path(nb_path), as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "code":
            exec(cell.source, globals())

MODEL = "llama3.1:8b"
TRACE_DIR = Path("traces"); TRACE_DIR.mkdir(exist_ok=True)
TRACE_FILE = TRACE_DIR / "agent_traces.jsonl"
print("trace file:", TRACE_FILE.resolve())

## 2. Tracing

Wrap `ollama.chat` so every call writes a JSONL record. Each record is enough to reproduce the call: messages in, messages out, tools available, options.

Side benefit: when something goes wrong in a 4-step run, you have a deterministic log of the whole thing instead of trying to reconstruct it from notebook stdout.

In [ ]:
def trace_call(record):
    """Append a single trace record to the JSONL file. Any non-serialisable
    field becomes its repr — we want a record, not a perfect round-trip."""
    with TRACE_FILE.open("a") as f:
        f.write(json.dumps(record, default=str) + "\n")

def chat_traced(*, run_id, step, messages, tools=None, model=MODEL, options=None):
    """A drop-in replacement for ollama.chat that records the call."""
    options = options or {"temperature": 0.2, "num_ctx": 4096}
    started = datetime.now(timezone.utc).isoformat()
    resp = ollama.chat(model=model, messages=messages, tools=tools, options=options)
    trace_call({
        "run_id": run_id,
        "step": step,
        "started_at": started,
        "finished_at": datetime.now(timezone.utc).isoformat(),
        "model": model,
        "messages_in": messages,
        "tools": [t["function"]["name"] for t in (tools or [])],
        "message_out": resp["message"],
    })
    return resp

## 3. Guardrails

Three layers, applied in `dispatch_tool_call_guarded`:

1. **Allowlist** — which tools is this run allowed to call? Forbid the rest, even if registered.
2. **Per-run budget** — total tool calls capped, per-run.
3. **Confirmation gate** — for tools that *mutate* state (write to DuckDB, hit external APIs that aren't read-only), require an explicit `confirm=True` flag from the caller. Without it, the tool refuses and returns an error the model can read.

In [ ]:
MUTATING_TOOLS = {"remember_note", "enqueue_task", "complete_task"}  # nb 04 + nb 05

class GuardError(Exception): pass

def make_guarded_dispatcher(*, allow=None, budget=10, auto_confirm_mutations=True):
    """Return a dispatch function bound to per-run state.
    `allow`: optional set of allowed tool names; None = all registered tools.
    `budget`: max tool calls before refusing.
    `auto_confirm_mutations`: in a notebook we set True; in production set False
      and route through human-in-the-loop confirmation."""
    state = {"calls": 0}
    def go(name, args):
        if allow is not None and name not in allow:
            return {"error": f"tool {name!r} not allowed in this run"}
        if state["calls"] >= budget:
            return {"error": f"tool budget exhausted ({budget})"}
        state["calls"] += 1
        if name in MUTATING_TOOLS and not auto_confirm_mutations:
            return {"error": f"{name} requires explicit human confirmation in this run"}
        return dispatch_tool_call(name, args)
    go.state = state
    return go

## 4. Guarded + traced loop

Same shape as nb 03's loop, but uses `chat_traced` and `make_guarded_dispatcher`. Run the same questions you used in nb 02–03 to populate the trace file.

In [ ]:
def truncate_for_model(payload, max_chars=1500):
    s = json.dumps(payload)
    return s if len(s) <= max_chars else s[:max_chars] + f"... [truncated, total {len(s)}]"

SYSTEM_PROMPT = (
    "You are a CTI analyst assistant. Use the available tools to answer. Stop "
    "calling tools as soon as you can answer. Do not repeat the same call."
)

def run_traced(question, *, run_id, allow=None, budget=8, max_steps=4):
    dispatch = make_guarded_dispatcher(allow=allow, budget=budget)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tool_use_count = Counter()
    for step in range(1, max_steps + 1):
        resp = chat_traced(run_id=run_id, step=step, messages=messages, tools=TOOL_SCHEMAS)
        msg = resp["message"]
        messages.append(msg)
        tool_calls = msg.get("tool_calls") or []
        if not tool_calls:
            return msg.get("content") or "", tool_use_count, dispatch.state["calls"]
        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = tc["function"].get("arguments") or {}
            if isinstance(fn_args, str):
                try: fn_args = json.loads(fn_args)
                except json.JSONDecodeError: fn_args = {}
            tool_use_count[fn_name] += 1
            result = dispatch(fn_name, fn_args)
            content = truncate_for_model(result)
            messages.append({"role": "tool", "name": fn_name, "content": content})
    return "(stopped: step cap)", tool_use_count, dispatch.state["calls"]

EVAL_QUESTIONS = [
    "How many documents are in the corpus, broken down by source?",
    "What's the most recent high-severity alert? Cite alert_id.",
    "How many ransomware victims were posted in the last 30 days?",
    "Find a Hacking-labeled document and summarize it.",
    # Stress test: the corpus has no answer, so it should call tools then concede.
    "Has BlackCat posted any new victims this week? Yes or no with the count.",
]

for i, q in enumerate(EVAL_QUESTIONS, 1):
    run_id = f"eval_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_q{i}"
    print(f"\n=== Q{i}: {q} ===")
    answer, used, n = run_traced(q, run_id=run_id)
    print("FINAL:", answer[:300])
    print(f"tool calls: {n}, breakdown: {dict(used)}")

## 5. Replay from trace

The point of tracing: re-walk every decision the model made, **without** spending another GPU-second. Useful for debugging, regression testing, and writing post-mortems.

In [ ]:
def load_traces():
    if not TRACE_FILE.exists():
        return []
    return [json.loads(l) for l in TRACE_FILE.read_text().splitlines() if l.strip()]

def summarise_run(records):
    by_run = {}
    for rec in records:
        by_run.setdefault(rec["run_id"], []).append(rec)
    for run_id, recs in by_run.items():
        recs.sort(key=lambda r: r["step"])
        n_tool_calls = sum(len(r["message_out"].get("tool_calls") or []) for r in recs)
        last_content = recs[-1]["message_out"].get("content") or ""
        print(f"\n[{run_id}] steps={len(recs)} tool_calls={n_tool_calls}")
        print("  final:", last_content[:200])

summarise_run(load_traces())

## 6. Failure-mode counter

Walk the trace file and count the bread-and-butter failures from nb 02–05:

- `cap_hit` — run terminated because of MAX_STEPS or budget.
- `repeated_call` — same tool with identical args two steps in a row.
- `unknown_tool` — model invented a tool name.
- `bad_json_args` — arguments arrived as a malformed string.
- `tool_returned_error` — dispatcher returned `{'error': ...}`.

Tracking these gives you a real signal-vs-noise read on your local model. Two of these (`repeated_call`, `unknown_tool`) drop sharply if you swap to a frontier model — useful evidence when discussing tradeoffs.

In [ ]:
def classify_failures(records):
    counts = Counter()
    by_run = {}
    for rec in records:
        by_run.setdefault(rec["run_id"], []).append(rec)
    for run_id, recs in by_run.items():
        recs.sort(key=lambda r: r["step"])
        last_call = None
        for rec in recs:
            for tc in rec["message_out"].get("tool_calls") or []:
                fn = tc["function"]["name"]
                args = tc["function"].get("arguments")
                if fn not in TOOL_FUNCTIONS:
                    counts["unknown_tool"] += 1
                if isinstance(args, str):
                    counts["bad_json_args"] += 1
                # repeated call detection: same name + same args as previous turn
                key = (fn, json.dumps(args, sort_keys=True, default=str))
                if key == last_call:
                    counts["repeated_call"] += 1
                last_call = key
        if recs and "(stopped" in (recs[-1]["message_out"].get("content") or ""):
            counts["cap_hit"] += 1
    return counts

fc = classify_failures(load_traces())
print("failure mode counts across all runs in trace file:")
for k, v in fc.most_common():
    print(f"  {k:20s} {v}")

## 7. Human-in-the-loop confirmation (sketch)

Mutating tools (`remember_note`, `enqueue_task`, `complete_task`) should not run unattended in production. The pattern: when a mutating tool is requested, the dispatcher pauses and asks for confirmation. In a notebook this is `input()`; in a real system it's a Slack approval, a queue, or a code-review-like UI.

We won't actually block here — instead we demonstrate the pattern so you can see how to wire it in. Set `auto_confirm_mutations=False` when constructing the dispatcher to opt in.

In [ ]:
# Synthetic demo of the no-confirm path: the dispatcher refuses, returns an error,
# and the model would see this and adjust.
strict = make_guarded_dispatcher(auto_confirm_mutations=False, budget=5)
print("strict dispatch of remember_note ->", strict("remember_note",
      {"topic": "test", "note": "this should be refused without confirmation"}))
print("strict dispatch of query_documents ->", strict("query_documents", {"limit": 1})["count"])

## 8. Where this leaves you

After this notebook, you have:

- A **tool registry** (nb 01) that any future agent in this project can plug into.
- A **single-step exchange** (nb 02) you can debug at the message level.
- A **planning loop** (nb 03) with hard caps that won't run away on a small model.
- A **memory layer** (nb 04) where stateless agent meets stateful database.
- A **two-agent pipeline** (nb 05) showing role isolation and DuckDB handoff.
- A **trace + eval** harness (this notebook) that turns model behaviour into something you can measure.

The 8 GB GPU never broke a sweat. Everything you built here is portable to a frontier model API; you'd swap `ollama.chat()` for an API call and most of the failure-mode counts would drop sharply.

